# 📘 Guide 14: Docker basics

**Who it's for:** containers are new to you, and you'd like to understand how GradeWise gets deployed. **Time:** about 25 minutes. Docker is optional for the course.

"It works on my machine" is a classic problem. A **container** packages your app *with* its Python version, libraries and settings, so it runs the same on your laptop, Asha's school server, or the cloud.

### 🎯 You'll learn
1. Images, containers and the Dockerfile
2. Installing Docker on Windows, Mac and Linux
3. Writing a Dockerfile for a uv project
4. Build, run, and pass in your `.env` safely
5. Running an app plus a service with Docker Compose

## Part 1: The vocabulary

| Term | Analogy |
|---|---|
| **Dockerfile** | the recipe |
| **Image** | the meal, frozen: built once from the recipe, can be copied anywhere |
| **Container** | a running portion of that meal: start, stop, delete, start a fresh one |
| **Registry** (Docker Hub, GHCR) | the supermarket freezer where images are shared |

Containers are not virtual machines: they share your computer's kernel, so they start in a second and are light.

## Part 2: Installing Docker

- **Windows:** install **Docker Desktop** (it uses WSL 2; the installer guides you). Run commands in PowerShell.
- **Mac:** install **Docker Desktop** (pick Apple Silicon or Intel).
- **Linux:** install **Docker Engine** with your distribution's instructions; add yourself to the `docker` group to avoid `sudo`.

Check it from a terminal:
```bash
docker --version
docker run hello-world
```

This notebook checks whether Docker is installed without starting anything:

In [ ]:
import shutil
import subprocess

docker = shutil.which("docker")
if docker:
    version = subprocess.run([docker, "--version"], capture_output=True, text=True).stdout.strip()
    print("✅", version)
else:
    print("ℹ️ Docker isn't installed (that's fine: it's optional). See Part 2 to install it.")

## Part 3: A Dockerfile for a uv project

```dockerfile
# Dockerfile
FROM python:3.12-slim

# 💡 copy the uv binary from its official image
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv

WORKDIR /app

# Install dependencies first: this layer is cached until pyproject/uv.lock change
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev --no-install-project

# Then copy the code
COPY gradewise ./gradewise
COPY app ./app
RUN uv sync --frozen --no-dev

EXPOSE 8000
CMD ["uv", "run", "python", "-m", "uvicorn", "app.api:app", "--host", "0.0.0.0", "--port", "8000"]
```

And a `.dockerignore`, so secrets and junk stay out of the image:

```
.env
.venv
output/
**/__pycache__
.git
```

Why copy `pyproject.toml` first? Docker caches each step. Changing your code then only re-runs the last two steps, not the slow install.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Keep secrets out of images</h2>
            <span style="color:#ef4444;">Never <code>COPY .env</code> into an image or bake API keys into a Dockerfile. Anyone who gets the image can read them. Pass secrets in when the container <i>starts</i> (Part 4).</span>
        </td>
    </tr>
</table>

## Part 4: Build and run

```bash
docker build -t gradewise .                               # build an image called gradewise
docker run --rm -p 8000:8000 --env-file .env gradewise    # run it, with your .env passed in at start-up
```

- `-p 8000:8000` maps the container's port to yours, so `http://localhost:8000/docs` works.
- `--env-file .env` passes your settings in at runtime.
- `--rm` deletes the container when it stops.

Saving output to your machine with a **volume**:
- **Windows (PowerShell):** `docker run --rm -v "${PWD}/output:/app/output" --env-file .env gradewise`
- **Mac / Linux:** `docker run --rm -v "$(pwd)/output:/app/output" --env-file .env gradewise`

Handy everyday commands (identical everywhere):

| Command | What it does |
|---|---|
| `docker ps` | running containers |
| `docker images` | images you've built or pulled |
| `docker logs <id>` | a container's output |
| `docker stop <id>` | stop a container |
| `docker system prune` | clean up unused stuff (asks first) |

## Part 5: Docker Compose: app + Ollama together

`compose.yaml` describes several containers that run together, here GradeWise's API plus a local Ollama model server:

```yaml
services:
  api:
    build: .
    ports: ["8000:8000"]
    env_file: .env
    environment:
      OLLAMA_BASE_URL: http://ollama:11434    # 💡 services find each other by name
    depends_on: [ollama]
  ollama:
    image: ollama/ollama
    volumes: ["ollama-models:/root/.ollama"]
volumes:
  ollama-models:
```

```bash
docker compose up --build     # start everything; Ctrl + C to stop
docker compose down           # remove the containers
```

Let's sanity-check the Compose file with Python (no Docker needed), the kind of check you might add to your tests:

In [ ]:
compose = {
    "services": {
        "api": {"build": ".", "ports": ["8000:8000"], "env_file": ".env", "depends_on": ["ollama"]},
        "ollama": {"image": "ollama/ollama", "volumes": ["ollama-models:/root/.ollama"]},
    },
    "volumes": {"ollama-models": None},
}

services = compose["services"]
for name, svc in services.items():
    missing = [d for d in svc.get("depends_on", []) if d not in services]
    source = svc.get("image") or f"built from {svc.get('build')}"
    print(f"{'✅' if not missing else '❌'} {name:<7} {source}" + (f"  missing: {missing}" if missing else ""))
print("🔐 .env passed at runtime, not baked in:", services["api"]["env_file"] == ".env")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Try it</h2>
            <span style="color:#f59e0b;">If you have Docker: write the Dockerfile for the mini API from Guide 13, build it, run it with <code>-p 8000:8000</code>, and open <code>/docs</code>. No Docker? Write the Dockerfile and <code>.dockerignore</code> anyway, and explain to a friend (or your AI assistant) what each line does.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Docker: get started: <a href="https://docs.docker.com/get-started/">docs.docker.com/get-started/</a><br/>• Using uv in Docker: <a href="https://docs.astral.sh/uv/guides/integration/docker/">docs.astral.sh/uv/guides/integration/docker/</a><br/>• Docker Compose: <a href="https://docs.docker.com/compose/">docs.docker.com/compose/</a></span>
        </td>
    </tr>
</table>